In [0]:
!pip install openpyxl

DEPRECATION: Using the pkg_resources metadata backend is deprecated. pip 26.3 will enforce this behaviour change. A possible replacement is to use the default importlib.metadata backend, by unsetting the _PIP_USE_IMPORTLIB_METADATA environment variable. Discussion can be found at https://github.com/pypa/pip/issues/13317
Looking in indexes: https://pypi.org/simple, https://****@pkgs.dev.azure.com/EHI-DataEngineeringPlatform/_packaging/EHI-DataEngineeringPlatform/pypi/simple/
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
schemas = [
    "ody_tb",
    "ecars_tb"

]

In [0]:
# %python
 
# table_name = 'appl_cntrl_log_l1'
# env="edl_dev"
 
# for i in schemas:
#     print(f'creating backup of : {i}.appl_cntrl_log_l1')
#     spark.sql(f"""CREATE OR REPLACE TABLE t30day_tb.{i}_appl_cntrl_log_l1_t30backup_{env}
#         AS SELECT * FROM {env}.{i}.{table_name}
#         """
#     )
   

creating backup of : ody_tb.appl_cntrl_log_l1
creating backup of : reference_tb.appl_cntrl_log_l1
creating backup of : ace_gdpr_tb.appl_cntrl_log_l1
creating backup of : app_security_tb.appl_cntrl_log_l1
creating backup of : arms_tb.appl_cntrl_log_l1
creating backup of : arms_auto_tb.appl_cntrl_log_l1
creating backup of : cam_tb.appl_cntrl_log_l1
creating backup of : cntct_ctr_tb.appl_cntrl_log_l1
creating backup of : csdp_tb.appl_cntrl_log_l1
creating backup of : dcls_veh_resdbuy_tb.appl_cntrl_log_l1
creating backup of : dss_metrics_tb.appl_cntrl_log_l1
creating backup of : ecars_tb.appl_cntrl_log_l1
creating backup of : ecir_tb.appl_cntrl_log_l1
creating backup of : fltsht_tb.appl_cntrl_log_l1
creating backup of : indiv_pref_tb.appl_cntrl_log_l1
creating backup of : lrd_tb.appl_cntrl_log_l1
creating backup of : osdwadmin_tb.appl_cntrl_log_l1
creating backup of : paymt_tb.appl_cntrl_log_l1
creating backup of : pbta_tb.appl_cntrl_log_l1
creating backup of : prism_tb.appl_cntrl_log_l1
c

In [0]:
from pyspark.sql.functions import lit
from functools import reduce

catalog = "edl_dev"
df_list = []

for schema in schemas:
    try:
        df = (
            spark.table(f"{catalog}.{schema}.appl_cntrl_log_l1")
            .withColumn("catalog_schema", lit(schema))
        )
        df_list.append(df)
    except:
        print(f"Skipping {schema}")

final_df = reduce(lambda x, y: x.unionByName(y), df_list)

# Replace literal "null" strings with actual nulls
clean_df = final_df.replace("null", None)

clean_df.createOrReplaceTempView("dev_df")

spark.sql("""
CREATE OR REPLACE TABLE t30day_tb.appl_cntrl_log_l1_t30backup_dev
USING DELTA
PARTITIONED BY (appl_cntrl_id)
AS SELECT * FROM dev_df
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
from pyspark.sql.functions import lit
from functools import reduce

catalog = "edl_prd"
df_list = []

for schema in schemas:
    try:
        df = (
            spark.table(f"{catalog}.{schema}.appl_cntrl_log_l1")
            .withColumn("catalog_schema", lit(schema))
        )
        df_list.append(df)
    except:
        print(f"Skipping {schema}")

# Union all schemas
final_df = reduce(lambda x, y: x.unionByName(y), df_list)

# Replace "null" string with actual null
clean_df = final_df.replace("null", None)

# Convert to pandas
pdf = clean_df.toPandas()

# Save to Excel
pdf.to_excel("prd_full.xlsx", index=False)

In [0]:
# =========================================
# Read Excel -> Create Spark DF -> Create Table with Fixed Metadata -> Load Data
# =========================================

import pandas as pd

# 1. Read Excel using pandas
prd_df1 = pd.read_excel("prd_full.xlsx")

# 2. Convert pandas DataFrame to Spark DataFrame
prd_df = spark.createDataFrame(prd_df1)

# 3. Replace string 'null' with actual null
clean_df = prd_df.replace("null", None)

# 4. Create temp view
clean_df.createOrReplaceTempView("prd_df")

# 5. Create NEW table using fixed metadata (NO CTAS)
spark.sql("""
CREATE OR REPLACE TABLE t30day_tb.appl_cntrl_log_l1_t30backup_prd (
  appl_cntrl_id INT,
  appl_cntrl_grp VARCHAR(256),
  appl_cntrl_nam VARCHAR(256),
  run_set SMALLINT,
  run_set_seq SMALLINT,
  load_nam VARCHAR(256),
  load_tool VARCHAR(25),
  versioned VARCHAR(1),
  trunc_reload VARCHAR(1),
  src_typ VARCHAR(256),
  src_sys VARCHAR(256),
  src_obj VARCHAR(256),
  src_fld_list_txt VARCHAR(8000),
  src_row_cnt BIGINT,
  stg_typ VARCHAR(256),
  stg_sys VARCHAR(256),
  stg_obj VARCHAR(256),
  stg_fld_list_txt VARCHAR(8000),
  stg_row_cnt BIGINT,
  tgt_typ VARCHAR(256),
  tgt_sys VARCHAR(256),
  tgt_obj VARCHAR(256),
  tgt_fld_list_txt VARCHAR(8000),
  tgt_row_cnt BIGINT,
  wmk_fld VARCHAR(256),
  data_as_of_tsp TIMESTAMP,
  param_strt_tsp TIMESTAMP,
  param_end_tsp TIMESTAMP,
  sql_str1 VARCHAR(8000),
  sql_str2 VARCHAR(8000),
  sql_str3 VARCHAR(8000),
  sql_str_lst_exectd VARCHAR(8000),
  prmry_key VARCHAR(256),
  order_by_fld VARCHAR(256),
  part_key VARCHAR(256),
  appl_strt_tsp TIMESTAMP,
  appl_end_tsp TIMESTAMP,
  status_cde VARCHAR(10),
  err_cde VARCHAR(8000),
  err_msg VARCHAR(8000),
  actv_flg INT,
  note_txt VARCHAR(8000),
  config_ext STRING,
  criticality INT,
  sla INT,
  hard_delete INT,
  domain STRING,
  dl_upd_tsp TIMESTAMP,
  dl_crte_tsp TIMESTAMP,
  catalog_schema STRING
)
USING DELTA
PARTITIONED BY (appl_cntrl_id)
TBLPROPERTIES (
  'delta.minReaderVersion' = '1',
  'delta.minWriterVersion' = '2'
)
""")

# 6. Load data into the table (uses metadata as-is)
spark.sql("""
INSERT INTO t30day_tb.appl_cntrl_log_l1_t30backup_prd
SELECT * FROM prd_df
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [ ]:
######In this below cell - exclude_columns - env related specific column names should be added.. 

In [0]:
from pyspark.sql.functions import col, when, lit, concat_ws, array, array_remove
from pyspark.sql.types import TimestampType

# =========================================
# Configurable Exclusion Columns (You can modify later)
# =========================================
exclude_columns = [
    "stg_sys",
    "note_txt",
    "config_ext",
    "actv_flg"
]

# =========================================
# Read Dev & PRD tables
# =========================================
dev_df = spark.table("t30day_tb.appl_cntrl_log_l1_t30backup_dev")
prd_df = spark.table("t30day_tb.appl_cntrl_log_l1_t30backup_prd")

# =========================================
# Identify timestamp columns dynamically (from DEV)
# =========================================
timestamp_cols = [
    field.name for field in dev_df.schema.fields
    if isinstance(field.dataType, TimestampType)
]

# =========================================
# Columns to exclude from comparison
# =========================================
exclude_all = set(timestamp_cols + exclude_columns + [
    "catalog_schema", "appl_cntrl_id", "tgt_sys"
])

# Columns to compare
compare_cols = [c for c in dev_df.columns if c not in exclude_all]

print("Timestamp Columns:", timestamp_cols)
print("Comparing Columns:", compare_cols)

# =========================================
# Join condition (Unique Key)
# =========================================
join_cond = [
    dev_df.catalog_schema == prd_df.catalog_schema,
    dev_df.appl_cntrl_id == prd_df.appl_cntrl_id,
    dev_df.tgt_sys == prd_df.tgt_sys
]

# =========================================
# Full Outer Join
# =========================================
joined_df = dev_df.alias("dev").join(
    prd_df.alias("prd"),
    on=join_cond,
    how="fullouter"
)

# =========================================
# Build Difference Columns Logic
# =========================================
diff_exprs = []

for c in compare_cols:
    diff_exprs.append(
        when(
            ~col(f"dev.{c}").eqNullSafe(col(f"prd.{c}")),
            lit(c)
        )
    )

diff_columns = array_remove(array(*diff_exprs), None)

# =========================================
# Status Logic
# =========================================
status_col = when(
    col("dev.appl_cntrl_id").isNull(),
    lit("MISSING_IN_DEV")
).when(
    col("prd.appl_cntrl_id").isNull(),
    lit("EXTRA_IN_DEV")
).when(
    size(diff_columns) > 0,
    lit("MISMATCH")
).otherwise(lit("MATCH"))

# =========================================
# Select Columns (Side-by-side)
# =========================================
select_expr = []

# Keys
select_expr.extend([
    col("prd.catalog_schema").alias("catalog_schema"),
    col("prd.appl_cntrl_id").alias("appl_cntrl_id"),
    col("prd.tgt_sys").alias("tgt_sys")
])

# Add PRD + DEV columns
for c in dev_df.columns:
    select_expr.append(col(f"prd.{c}").alias(f"prd_{c}"))
    select_expr.append(col(f"dev.{c}").alias(f"dev_{c}"))

# Final DF
final_df = joined_df.select(*select_expr) \
    .withColumn("status", status_col) \
    .withColumn("diff_columns", concat_ws(",", diff_columns))

# =========================================
# Create Final Table
# =========================================
final_df.createOrReplaceTempView("final_view")

spark.sql("""
CREATE OR REPLACE TABLE t30day_tb.final_table
USING DELTA
PARTITIONED BY (appl_cntrl_id)
AS
SELECT * FROM final_view
""")

num_affected_rows,num_inserted_rows


In [ ]:
######In this below cell - exclude_columns - env related specific column names should be added.. 

In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType

# =========================================
# Config
# =========================================
exclude_columns = [
    "stg_sys",
    "note_txt",
    "config_ext",
    "actv_flg"
]

final_df = spark.table("t30day_tb.final_table")

# Get all schemas dynamically
schemas = [row['catalog_schema'] for row in final_df.select("catalog_schema").distinct().collect()]

for schema in schemas:
    print(f"Processing schema: {schema}")
    
    table_name = f"t30day_tb.{schema}_appl_cntrl_log_l1_t30backup_edl_dev"
    
    dev_table = spark.table(table_name)
    
    # =========================================
    # Identify timestamp columns from DEV TABLE
    # =========================================
    timestamp_cols = [
        f.name for f in dev_table.schema.fields
        if isinstance(f.dataType, TimestampType)
    ]
    
    # Columns in actual backup table (IMPORTANT)
    target_cols = dev_table.columns
    
    # Remove tracking columns (if any not in actual table)
    usable_cols = [c for c in target_cols if c not in ["catalog_schema"]]
    
    # =========================================
    # Filter relevant records
    # =========================================
    sub_final = final_df.filter(col("catalog_schema") == schema)
    
    # =========================================
    # Prepare SOURCE (PRD values)
    # =========================================
    src_df = sub_final.select(
        *[col(f"prd_{c}").alias(c) for c in usable_cols if f"prd_{c}" in sub_final.columns],
        "status"
    )
    
    src_df.createOrReplaceTempView("src_view")
    dev_table.createOrReplaceTempView("tgt_view")
    
    # =========================================
    # Build dynamic SET clause (UPDATE)
    # =========================================
    set_expr = []
    
    for c in usable_cols:
        if c in ["appl_cntrl_id", "tgt_sys"]:
            continue
        
        if c in exclude_columns:
            set_expr.append(f"{c} = NULL")
        elif c in timestamp_cols:
            set_expr.append(f"{c} = current_timestamp()")
        else:
            set_expr.append(f"{c} = src.{c}")
    
    set_expr.append("actv_flg = 0")
    
    set_clause = ",\n".join(set_expr)
    
    # =========================================
    # UPDATE (MISMATCH)
    # =========================================
    spark.sql(f"""
    MERGE INTO {table_name} tgt
    USING (
        SELECT * FROM src_view WHERE status = 'MISMATCH'
    ) src
    ON tgt.appl_cntrl_id = src.appl_cntrl_id
       AND tgt.tgt_sys = src.tgt_sys
    WHEN MATCHED THEN UPDATE SET
    {set_clause}
    """)
    
    # =========================================
    # INSERT (MISSING_IN_DEV)
    # =========================================
    
    insert_cols = []
    insert_vals = []
    
    for c in usable_cols:
        insert_cols.append(c)
        
        if c in exclude_columns:
            insert_vals.append("NULL")
        elif c in timestamp_cols:
            insert_vals.append("current_timestamp()")
        else:
            insert_vals.append(f"src.{c}")
    
    insert_cols.append("actv_flg")
    insert_vals.append("0")
    
    insert_cols_str = ", ".join(insert_cols)
    insert_vals_str = ", ".join(insert_vals)
    
    spark.sql(f"""
    INSERT INTO {table_name}
    ({insert_cols_str})
    SELECT {insert_vals_str}
    FROM src_view src
    WHERE status = 'MISSING_IN_DEV'
    """)

In [0]:
from pyspark.sql.functions import col

final_df = spark.table("t30day_tb.final_table")

schemas = [row['catalog_schema'] for row in final_df.select("catalog_schema").distinct().collect()]

for schema in schemas:
    print(f"Restoring actv_flg for schema: {schema}")
    
    table_name = f"t30day_tb.{schema}_appl_cntrl_log_l1_t30backup_edl_dev"
    
    sub_final = final_df.filter(col("catalog_schema") == schema)
    
    sub_final.createOrReplaceTempView("final_view")
    
    spark.sql(f"""
    MERGE INTO {table_name} tgt
    USING (
        SELECT 
            appl_cntrl_id,
            tgt_sys,
            status,
            dev_actv_flg,
            prd_actv_flg
        FROM final_view
    ) src
    ON tgt.appl_cntrl_id = src.appl_cntrl_id
       AND tgt.tgt_sys = src.tgt_sys
    
    WHEN MATCHED AND src.status = 'MISMATCH' THEN
      UPDATE SET tgt.actv_flg = src.dev_actv_flg
    
    WHEN MATCHED AND src.status = 'MISSING_IN_DEV' THEN
      UPDATE SET tgt.actv_flg = src.prd_actv_flg
    """)